<a href="https://colab.research.google.com/github/Saryace/rusle-ren237/blob/main/rusle_ren237.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este será el proyecto base que usaremos para crear un modelo RUSLE usando Google Earth Engine

In [24]:
import math
import os

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

**Pasos para activar su cuenta de Google con un proyecto GEE**

1. Crear cuenta de Google (si no tienen una)

2. Ir a https://earthengine.google.com/signup/ e iniciar sesión con su cuenta de Google. Completar el formulario de registro seleccionando 'Uso no comercial / académico'
3. Crear un proyecto en Google Cloud Console
Ir a https://console.cloud.google.com/projectcreate, ponerle un nombre al proyecto (ej. 'clase-erosion-nombre') y anotar ese Project ID
4. Habilitar la API de Earth Engine en el proyecto
Con el proyecto creado y seleccionado, ir a https://console.cloud.google.com/apis/library/earthengine.googleapis.com y hacer clic en 'Habilitar'
5. Registrar el proyecto para uso con Earth Engine
Ir a https://console.cloud.google.com/earth-engine/configuration y seguir los indicando que es para investigación/uso académico no comercial.


In [25]:
GEE_PROJECT = 'ee-seaceved'

import ee

print("Autenticando...")
ee.Authenticate()

print(f"Inicializando proyecto: {GEE_PROJECT}...")
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✓ Google Earth Engine inicializado correctamente!")

    info = ee.Number(1).getInfo()
    print("✓ Conexión verificada.")

except ee.EEException as e:
    print(f"\n✗ Falló la inicialización: {e}")
    if "not registered" in str(e):
        print(f"\nACCIÓN REQUERIDA: El proyecto '{GEE_PROJECT}' no está registrado en Earth Engine.")
        print(f"1. Regístralo aquí: https://console.cloud.google.com/earth-engine/configuration?project={GEE_PROJECT}")
        print("2. O actualiza la variable GEE_PROJECT con un Project ID válido.")
    elif "permission" in str(e).lower():
        print(f"\nACCIÓN REQUERIDA: Puede que no tengas permisos para el proyecto '{GEE_PROJECT}'.")
        print("Revisa la configuración de IAM en Google Cloud Console.")
except Exception as e:
    print(f"\n✗ Ocurrió un error inesperado: {e}")

Autenticando...
Inicializando proyecto: ee-seaceved...
✓ Google Earth Engine inicializado correctamente!
✓ Conexión verificada.


In [26]:
# Leemos una cuenca del inventario de erosion de CIREN
# Yo elegi la del Estero Cholqui https://inventarioerosion.ciren.cl/layers/geonode_data:geonode:r13_168_rep_1ha
import geopandas as gpd

url = "https://inventarioerosion.ciren.cl/geoserver/ows?service=WFS&version=1.0.0&request=GetFeature&typename=geonode%3Ar13_168_rep_1ha&outputFormat=json&srs=EPSG%3A32719&srsName=EPSG%3A32719"
gp_df = gpd.read_file(url)
gp_df_simple = gp_df.dissolve() # opcion de geopandas para solo tener el contorno de la cuenca
area_de_interes = geemap.geopandas_to_ee(gp_df_simple) # de geopandas a gee


In [27]:
# Visualizamos el area de interes
mapa_simple = geemap.Map()
mapa_simple.centerObject(area_de_interes, zoom=12)
mapa_simple.addLayer(area_de_interes, {'color': 'black'}, 'Área de interés')
mapa_simple

Map(center=[-33.809034207791065, -71.08282686452921], controls=(WidgetControl(options=['position', 'transparen…

Vamos a comenzar con el primer componente de RUSLE: erosividad de la lluvia.Según Bonilla y Vidal 2011, R = 0.0028P^1.534, donde P es la precipitación anual en mm. Vamos a cargar las precipitaciones de un año hidrológico (inicia 1 de abril y termina el 31 de marzo) solo como ejemplo para recordar el concepto. Pero por lo general los estudios son anuales (1 enero a 1 de diciembre)


In [28]:
# 1. Precipitación acumulada del año hidrológico (CHIRPS diario, ~5,5 km)
#    El año hidrológico chileno va del 1 de abril al 31 de marzo.
precipitacion = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                 .filterBounds(area_de_interes)
                 .filterDate('2023-04-01', '2024-04-01')
                 .select('precipitation'))

P_anual = precipitacion.sum().clip(area_de_interes).rename('P_mm')

# 2. R = 0.028 * P^1.534   (Bonilla & Vidal 2011) -> MJ mm ha-1 h-1 an-1
R = P_anual.pow(1.534).multiply(0.028).rename('R')

# 3. Traer los valores desde el servidor a Python
est_R = R.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes.geometry(),
    scale=5566,                # resolución nativa de CHIRPS
    maxPixels=1e9,
    bestEffort=True).getInfo()

print("Rango de R:", {k: round(v) for k, v in est_R.items()})

# 4. Visualizar con el rango tomado de los propios datos
viz_R = {'min': est_R['R_min'], 'max': est_R['R_max'],
         'palette': ['white', 'yellow', 'orange', 'red']}

mapa_simple.addLayer(R, viz_R, 'Erosividad R')
mapa_simple

Rango de R: {'R_max': 273, 'R_min': 232}


Map(bottom=629333.0, center=[-33.809034207791065, -71.08282686452921], controls=(WidgetControl(options=['posit…

Ahora vamos a descargar datos de suelo (arena, limo y arcilla + materia orgánica y usar la fórmula de EPIC para obtener valores de K. Los obtendremos de OpenLandMap https://stac.openlandmap.org/organic.carbon_usda.6a1c/collection.json?.language=en

In [30]:

# Carbono orgánico del suelo (g/kg, capa 0 cm)
carbono_organico = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arcilla (% en masa, capa 0 cm)
arcilla = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arena (% en masa, capa 0 cm)
arena = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Limo (% en masa, capa 0 cm)
limo = ee.Image(100).subtract(arena).subtract(arcilla)


In [32]:
carbono_organico_pct = carbono_organico.multiply(0.5)  # OpenLandMap: b0 x 5 g/kg → %

sn1 = ee.Image(1).subtract(arena.divide(100))  # fraccion que no es arena

# f(csand) = 0.2 + 0.3*exp[-0.0256*arena*(1 - limo/100)]
factor1 = ee.Image(0.2).add(
    ee.Image(0.3).multiply(
        arena.multiply(-0.0256).multiply(
            ee.Image(1).subtract(limo.divide(100))
        ).exp()
    )
)

# f(cl-si) = (limo / (arcilla + limo))^0.3
factor2 = (limo.divide(arcilla.add(limo))).pow(0.3)

# f(orgc) = 1 - 0.25*orgC / (orgC + exp[3.72 - 2.95*orgC])
factor3 = ee.Image(1).subtract(
    ee.Image(0.25).multiply(carbono_organico_pct).divide(
        carbono_organico_pct.add(
            (ee.Image(3.72).subtract(carbono_organico_pct.multiply(2.95))).exp()
        )
    )
)

# f(hisand) = 1 - 0.7*SN1 / (SN1 + exp[-5.51 + 22.9*SN1])
factor4 = ee.Image(1).subtract(
    ee.Image(0.7).multiply(sn1).divide(
        sn1.add((ee.Image(-5.51).add(sn1.multiply(22.9))).exp())
    )
)

# K primero se define, despues se reduce
K_us = factor1.multiply(factor2).multiply(factor3).multiply(factor4)  # unidades EPIC
K = K_us.multiply(0.1317).rename('K')   # a SI, para calzar con el R de Bonilla & Vidal

stats_K = K.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes.geometry(),
    scale=250,              # resolucion nativa de OpenLandMap
    maxPixels=1e9,
    bestEffort=True
).getInfo()

print("Rango de K en SI:", {k: round(v, 4) for k, v in stats_K.items()})

viz_K = {'min': stats_K['K_min'], 'max': stats_K['K_max'],
         'palette': ['white', 'lightgreen', 'green', 'darkgreen']}
mapa_simple.addLayer(K, viz_K, 'Erodabilidad K')
mapa_simple

Rango de K en SI: {'K_max': 0.0302, 'K_min': 0.0225}


Map(bottom=629333.0, center=[-33.809034207791065, -71.08282686452921], controls=(WidgetControl(options=['posit…

---
## 6. Factores L y S — Topografía

$$L = \left(\frac{\lambda}{22.13}\right)^m \qquad S = \frac{0.43 + 0.30\,s + 0.043\,s^2}{6.613}$$

con $\lambda$ = longitud de pendiente (m) y $s$ = pendiente (%). El 22,13 es la longitud de la parcela estándar de la USLE (72,6 pies), así que $L = 1$ cuando la ladera mide exactamente eso.

### El exponente $m$

$m$ controla **qué tan rápido crece la erosión al alargarse la ladera**, y no es constante: depende de la pendiente misma. La razón es física. En laderas tendidas domina la erosión entre surcos (*interrill*), producida por el impacto de la gota de lluvia, que ocurre de manera pareja en toda la superficie y por lo tanto casi no depende del largo. En laderas empinadas el flujo se concentra en surcos (*rill*) que ganan volumen y velocidad ladera abajo, de modo que cada metro adicional aporta más erosión que el anterior.

Wischmeier & Smith (1978) tabularon $m$ según la razón entre erosión en surcos y entre surcos:

| Pendiente $s$ | $m$ | Proceso dominante |
|---|---|---|
| < 1 % | 0,2 | *interrill* — el largo casi no importa |
| 1 – 3 % | 0,3 | transición |
| 3 – 4,5 % | 0,4 | transición |
| > 4,5 % | 0,5 | *rill* — el largo importa mucho |


In [33]:
dem = ee.Image('USGS/SRTMGL1_003').clip(area_de_interes)

TAMANO_PIXEL_L  = 30.0

pendiente_grados = ee.Terrain.slope(dem)
pendiente_pct = pendiente_grados.multiply(3.14159265 / 180).tan().multiply(100).rename('pendiente_pct')

# Exponente m según la pendiente (Wischmeier & Smith 1978)
m = (ee.Image(0.5)
     .where(pendiente_pct.lte(1.0), 0.2)
     .where(pendiente_pct.gt(1.0).And(pendiente_pct.lte(3.0)), 0.3)
     .where(pendiente_pct.gt(3.0).And(pendiente_pct.lte(4.5)), 0.4)
     .where(pendiente_pct.gt(4.5), 0.5)
     .rename('m'))

lam = pendiente_pct.multiply(0).add(TAMANO_PIXEL_L)   # raster constante = tamaño de píxel

L = lam.divide(22.13).pow(m).rename('L')

S = (pendiente_pct.pow(2).multiply(0.043)
     .add(pendiente_pct.multiply(0.30))
     .add(0.43)
     .divide(6.613)
     .rename('S'))


De Jong (1994) relacionó el factor $C$ de campo con el NDVI:

$$C = 0.431 - 0.805 \cdot NDVI$$


In [18]:
def escalar_l8(img):
    """Aplica los factores de escala de Landsat Collection 2 Level 2."""
    optico = img.select('SR_B.').multiply(0.0000275).add(-0.2)
    return img.addBands(optico, None, True)

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(area_de_interes)
      .filterDate('2023-04-01', '2024-04-01')
      .filter(ee.Filter.lt('CLOUD_COVER', 30))
      .map(escalar_l8))

print("Escenas Landsat 8 disponibles:", l8.size().getInfo())

compuesto = l8.median().clip(area_de_interes)
ndvi = compuesto.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

C = ee.Image(0.431).subtract(ndvi.multiply(0.805)).clamp(0, 1).rename('C')

Escenas Landsat 8 disponibles: 30


Factor P: Prácticas de conservación

Sin un catastro de prácticas de conservación, se asigna un valor de $P$ por clase de cobertura de suelo MODIS, siguiendo Chuenchum et al. (2019). **Chuenchum, P., Xu, M., & Tang, W. (2019).** Estimation of soil erosion and sediment yield in the Lancang–Mekong River using the modified RUSLE. *Water*, 12(1), 135.https://www.mdpi.com/2073-4441/12/1/135


In [8]:
VALORES_P = {
    1: 0.8, 2: 0.8, 3: 0.8, 4: 0.8, 5: 0.8,    # bosques
    6: 0.8, 7: 0.8, 8: 0.8, 9: 0.8, 10: 0.8,   # matorrales, sabanas, praderas
    11: 1.0,                                    # humedales
    12: 0.5,                                    # cultivos
    13: 1.0,                                    # urbano
    14: 0.5,                                    # mosaico cultivo / vegetación natural
    15: 1.0, 16: 1.0, 17: 1.0,                  # nieve, suelo desnudo, agua
}

cobertura = (ee.ImageCollection('MODIS/061/MCD12Q1')
             .select('LC_Type1')
             .sort('system:time_start', False)
             .first()
             .clip(area_de_interes)
             .rename('lulc'))

expresion = ": ".join(f"(b('lulc') == {c}) ? {v}" for c, v in VALORES_P.items()) + ": 1.0"
P = cobertura.expression(expresion).rename('P')
